MULTİCOLLİNEARİTY ANALİZİ

In [ ]:
# Veri okuma

In [ ]:
from google.colab import files
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# ===== Upload file =====
uploaded = files.upload()

# Dosya adını otomatik al
file_name = list(uploaded.keys())[0]

# ===== Kolon isimleri =====
feature_names = [
    "Jitter_local", "Jitter_local_abs", "Jitter_rap", "Jitter_ppq5", "Jitter_ddp",
    "Shimmer_local", "Shimmer_local_dB", "Shimmer_apq3", "Shimmer_apq5", "Shimmer_apq11", "Shimmer_dda",
    "AC", "NTH", "HTN",
    "Median_pitch", "Mean_pitch", "Pitch_std", "Min_pitch", "Max_pitch",
    "Num_pulses", "Num_periods", "Mean_period", "Period_std",
    "Frac_unvoiced_frames", "Num_voice_breaks", "Degree_voice_breaks"
]

cols = ["SubjectId"] + feature_names + ["UPDRS", "Class"]

# ===== Veri okuma =====
df = pd.read_csv(file_name, header=None, names=cols)

print("Dataset shape:", df.shape)
df.head()

Saving train_data.txt to train_data.txt
Dataset shape: (1040, 29)


,SubjectId,Jitter_local,Jitter_local_abs,Jitter_rap,Jitter_ppq5,Jitter_ddp,Shimmer_local,Shimmer_local_dB,Shimmer_apq3,Shimmer_apq5,...,Max_pitch,Num_pulses,Num_periods,Mean_period,Period_std,Frac_unvoiced_frames,Num_voice_breaks,Degree_voice_breaks,UPDRS,Class
0,1,1.488,0.000090,0.900,0.794,2.699,8.334,0.779,4.517,4.609,...,187.576,160,159,0.006065,0.000416,0.000,0,0.000,23,1
1,1,0.728,0.000038,0.353,0.376,1.059,5.864,0.642,2.058,3.180,...,234.505,170,169,0.005181,0.000403,2.247,0,0.000,23,1
2,1,1.220,0.000074,0.732,0.670,2.196,8.719,0.875,4.347,5.166,...,211.442,1431,1427,0.006071,0.000474,10.656,1,0.178,23,1
3,1,2.502,0.000123,1.156,1.634,3.469,13.513,1.273,5.263,8.771,...,220.230,94,92,0.004910,0.000320,0.000,0,0.000,23,1
4,1,3.509,0.000167,1.715,1.539,5.145,9.112,1.040,3.102,4.927,...,225.162,117,114,0.004757,0.000380,18.182,1,13.318,23,1


In [ ]:
X = df[feature_names]

corr_matrix = X.corr()

threshold = 0.90
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

high_corr = (
    upper.stack()
    .reset_index()
    .rename(columns={"level_0":"Feature_A","level_1":"Feature_B",0:"Correlation"})
)

high_corr["AbsCorr"] = high_corr["Correlation"].abs()
high_corr = high_corr[high_corr["AbsCorr"] >= threshold].sort_values("AbsCorr", ascending=False)

print("High correlation pairs (>|0.90|):")
high_corr

High correlation pairs (>|0.90|):


,Feature_A,Feature_B,Correlation,AbsCorr
156,Shimmer_apq3,Shimmer_dda,1.000000,1.000000
50,Jitter_rap,Jitter_ddp,1.000000,1.000000
304,Num_pulses,Num_periods,0.999108,0.999108
220,AC,NTH,-0.989425,0.989425
115,Shimmer_local,Shimmer_local_dB,0.963551,0.963551
1,Jitter_local,Jitter_rap,0.955695,0.955695
3,Jitter_local,Jitter_ddp,0.955694,0.955694
259,Median_pitch,Mean_pitch,0.930894,0.930894
275,Mean_pitch,Mean_period,-0.916771,0.916771
72,Jitter_ppq5,Jitter_ddp,0.915695,0.915695


In [ ]:
# |correlation| ≥ 0.90 olan feature çiftlerini filtreledik.
# Shimmer_apq3 – Shimmer_dda = 1.000 : Bu değişkenler matematiksel olarak birbirinden türetilmiş.
# Jitter_rap – Jitter_ddp = 1.000 : Bu değişkenler matematiksel olarak birbirinden türetilmiş.
# AC - NTH = -0.989425 : Neredeyse ters yönlü aynı bilgiyi taşıyorlar


In [ ]:
# VIF ÖLÇÜMÜ
# Her bir feature’ın, diğer feature’lar tarafından ne kadar lineer olarak açıklanabildiğini ölçer.
# Her feature için şunu sorduk:
# "Bu değişken, diğer değişkenlerin lineer kombinasyonu ile ne kadar iyi tahmin edilebiliyor?”
# “Bu feature, diğer feature’lar tarafından ne kadar açıklanabiliyor?”

# VIF ≈ 1 → bağımsız

# VIF 2–5 → hafif ilişki

# VIF 5–10 → dikkat

# VIF > 10 → ciddi multicollinearity

# VIF çok büyük (1000, milyon) → neredeyse kopya
# Yani VIF yüksekse: Bu feature neredeyse tamamen diğerlerinden hesaplanabiliyor.

In [ ]:
def compute_vif(X_df):
    X_const = sm.add_constant(X_df)
    vif_data = []

    for i in range(1, X_const.shape[1]):  # const hariç
        vif = variance_inflation_factor(X_const.values, i)
        vif_data.append((X_const.columns[i], vif))

    vif_df = pd.DataFrame(vif_data, columns=["Feature", "VIF"])
    return vif_df.sort_values("VIF", ascending=False)

vif_table = compute_vif(X)
vif_table

,Feature,VIF
10,Shimmer_dda,1.254011e+08
7,Shimmer_apq3,1.253993e+08
2,Jitter_rap,1.350349e+07
4,Jitter_ddp,1.350315e+07
19,Num_pulses,1.995182e+03
20,Num_periods,1.981530e+03
11,AC,1.455578e+02
12,NTH,9.928470e+01
15,Mean_pitch,4.818843e+01
0,Jitter_local,3.430855e+01


In [ ]:
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
vif_table

,Feature,VIF
10,Shimmer_dda,"125,401,056.686"
7,Shimmer_apq3,"125,399,268.938"
2,Jitter_rap,"13,503,488.215"
4,Jitter_ddp,"13,503,152.756"
19,Num_pulses,"1,995.182"
20,Num_periods,"1,981.530"
11,AC,145.558
12,NTH,99.285
15,Mean_pitch,48.188
0,Jitter_local,34.309


In [ ]:
def iterative_vif(X_df, limit=10):
    X_work = X_df.copy()
    dropped = []

    while True:
        vif_df = compute_vif(X_work)
        max_vif = vif_df["VIF"].max()

        if max_vif <= limit:
            break

        worst_feature = vif_df.iloc[0]["Feature"]
        dropped.append(worst_feature)
        X_work = X_work.drop(columns=[worst_feature])

    return X_work, dropped

X_reduced, dropped_features = iterative_vif(X, limit=10)

print("Dropped features:", dropped_features)
print("Remaining features:", list(X_reduced.columns))

Dropped features: ['Shimmer_dda', 'Jitter_rap', 'Num_pulses', 'AC', 'Mean_pitch', 'Jitter_local', 'Shimmer_local', 'Pitch_std']
Remaining features: ['Jitter_local_abs', 'Jitter_ppq5', 'Jitter_ddp', 'Shimmer_local_dB', 'Shimmer_apq3', 'Shimmer_apq5', 'Shimmer_apq11', 'NTH', 'HTN', 'Median_pitch', 'Min_pitch', 'Max_pitch', 'Num_periods', 'Mean_period', 'Period_std', 'Frac_unvoiced_frames', 'Num_voice_breaks', 'Degree_voice_breaks']


In [ ]:
# İteratif VIF
# VIF hesaplandı
# En yüksek VIF’li değişkeni silindi
# VIF yeniden hesaplandı
# Bu işlem VIF ≤ 10 olana kadar tekrarlandı